In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

from sys import path
path.append("./src/")
from model_3d import CosmoCNN

# Check GPU

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device, flush=True)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        flush=True
    )

Using device: cpu


# location settings

In [3]:
ROOT = "/Users/hyp0515/data/a3/CAMELS_multifield/"
M_TNG_FILE = ROOT + "Maps_Mtot_IllustrisTNG_LH_z=0.00.npy"
P_TNG_FILE = ROOT + "Maps_P_IllustrisTNG_LH_z=0.00.npy"
PARAM_TNG_FILE = ROOT + "params_LH_IllustrisTNG.txt"

M_SIMBA_FILE = ROOT + "Maps_Mtot_SIMBA_LH_z=0.00.npy"
P_SIMBA_FILE = ROOT + "Maps_P_SIMBA_LH_z=0.00.npy"
PARAM_SIMBA_FILE = ROOT + "params_LH_SIMBA.txt"




# ROOT = "./"
# M_TNG_FILE = ROOT + "data/Maps_Mtot_IllustrisTNG_LH_z=0.00.npy"
# P_TNG_FILE = ROOT + "data/Maps_P_IllustrisTNG_LH_z=0.00.npy"
# PARAM_TNG_FILE = ROOT + "data/params_LH_IllustrisTNG.txt"

# M_SIMBA_FILE = ROOT + "data/Maps_Mtot_SIMBA_LH_z=0.00.npy"
# P_SIMBA_FILE = ROOT + "data/Maps_P_SIMBA_LH_z=0.00.npy"
# PARAM_SIMBA_FILE = ROOT + "data/params_LH_SIMBA.txt"



NUM_EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

N_SIM = 1000
N_MAPS_PER_SIM = 15

RANDOM_SEED = 42
rng = np.random.default_rng(
    RANDOM_SEED
)



# Dataset

In [4]:
class CAMELSDataset(Dataset):

    def __init__(
        self,
        maps,
        params,
        sim_indices,
        x_mean,
        x_std
    ):

        self.maps = maps
        self.params = params
        self.sim_indices = sim_indices

        self.x_mean = x_mean
        self.x_std = x_std


    def __len__(self):

        return (
            len(self.sim_indices)
            * N_MAPS_PER_SIM
        )


    def __getitem__(self, idx):

        sim_local = (
            idx // N_MAPS_PER_SIM
        )

        map_id = (
            idx % N_MAPS_PER_SIM
        )

        sim_id = self.sim_indices[
            sim_local
        ]

        x = self.maps[
            sim_id,
            map_id
        ]

        y = self.params[
            sim_id
        ]


        # Normalize map
        x = (
            x - self.x_mean
        ) / self.x_std


        # Convert to tensor
        x = torch.as_tensor(
            np.asarray(x),
            dtype=torch.float32
        )

        y = torch.as_tensor(
            y,
            dtype=torch.float32
        )

        return x, y

# Load data

In [5]:
M_TNG = np.load(M_TNG_FILE, mmap_mode="r").reshape(
    N_SIM,
    N_MAPS_PER_SIM,
    256,
    256
)
P_TNG = np.load(P_TNG_FILE, mmap_mode="r").reshape(
    N_SIM,
    N_MAPS_PER_SIM,
    256,
    256
)

MP_TNG = np.stack((M_TNG, P_TNG), axis=2)
cosmos_params_TNG = np.loadtxt(PARAM_TNG_FILE)


M_SIMBA = np.load(M_SIMBA_FILE, mmap_mode="r").reshape(
    N_SIM,
    N_MAPS_PER_SIM,
    256,
    256
)
P_SIMBA = np.load(P_SIMBA_FILE, mmap_mode="r").reshape(
    N_SIM,
    N_MAPS_PER_SIM,
    256,
    256
)

MP_SIMBA = np.stack((M_SIMBA, P_SIMBA), axis=2)
cosmos_params_SIMBA = np.loadtxt(PARAM_SIMBA_FILE)




combine_idx = np.random.randint(0, N_SIM, size=500)
MP_COM = np.concatenate((MP_TNG[combine_idx], MP_SIMBA[combine_idx]), axis=0)
cosmos_params_COM = np.concatenate((cosmos_params_TNG[combine_idx], cosmos_params_SIMBA[combine_idx]), axis=0)


MP_list = [MP_TNG, MP_SIMBA, MP_COM]
cosmos_params_list = [cosmos_params_TNG, cosmos_params_SIMBA, cosmos_params_COM]
result_list = ['TNG', 'SIMBA', 'COM']

# Create output directory

In [6]:
i = 0
lin_or_log = 'LOG'



if lin_or_log == 'LOG':
    MP_ = np.log10(MP_list[i])
else:
    MP_ = MP_list[i]
cosmos_params = cosmos_params_list[i]
result = result_list[i]

In [7]:
RESULT_ROOT = "/Users/hyp0515/Desktop/a3_project/new/"
# RESULT_ROOT = '.'
RESULT_DIR = RESULT_ROOT + f"results/{result}_MP_BATCHSIZE{BATCH_SIZE}_{lin_or_log}/run_001"


os.makedirs(RESULT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    RESULT_DIR,
    "best_model.pt"
)

NORM_PATH = os.path.join(
    RESULT_DIR,
    "normalization.npz"
)

SPLIT_PATH = os.path.join(
    RESULT_DIR,
    "split_indices.npz"
)

HISTORY_PATH = os.path.join(
    RESULT_DIR,
    "loss_history.npz"
)


In [8]:
# shrink_step = 10
# N_SIM = N_SIM // shrink_step
# MP_ = MP_[::shrink_step, :, :, :, :]
# cosmos_params = cosmos_params[::shrink_step, :]


# print(
#     "Map shape:",
#     MP_.shape,
#     flush=True
# )

# print(
#     "Params shape:",
#     cosmos_params.shape,
#     flush=True
# )


# Train / validation / test split

In [9]:
idx = rng.permutation(
    N_SIM
)

train_idx = idx[:8*N_SIM//10]
val_idx   = idx[8*N_SIM//10:9*N_SIM//10]
test_idx  = idx[9*N_SIM//10:]


# Save split indices immediately
np.savez(
    SPLIT_PATH,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

# Normalize output parameters

In [10]:
# Apply log10 to all parameters except omega_m and sigma_8
cosmos_params[:, 2:] = np.log10(cosmos_params[:, 2:])


y_mean = cosmos_params[train_idx].mean(axis=0, keepdims=True)
y_std = cosmos_params[train_idx].std(axis=0, keepdims=True)
params_n = (cosmos_params - y_mean) / y_std


# Normalize input images
To avoid creating a huge temporary copy, calculate mean/std simulation by simulation.

In [11]:
total_sum = 0.0
total_sq_sum = 0.0
total_count = 0

for sim_id in train_idx:

    x = np.asarray(
        MP_[sim_id],
        dtype=np.float64
    )

    total_sum += x.sum()
    total_sq_sum += np.square(x).sum()
    total_count += x.size


x_mean = total_sum / total_count
x_var = (total_sq_sum / total_count - x_mean**2)
x_std = np.sqrt(x_var)



# Save normalization
np.savez(
    NORM_PATH,
    x_mean=x_mean,
    x_std=x_std,
    y_mean=y_mean,
    y_std=y_std
)

# Create datasets

In [12]:
train_dataset = CAMELSDataset(
    MP_,
    params_n,
    train_idx,
    x_mean,
    x_std
)

val_dataset = CAMELSDataset(
    MP_,
    params_n,
    val_idx,
    x_mean,
    x_std
)

test_dataset = CAMELSDataset(
    MP_,
    params_n,
    test_idx,
    x_mean,
    x_std
)

print(
    "Train maps:",
    len(train_dataset),
    flush=True
)

print(
    "Validation maps:",
    len(val_dataset),
    flush=True
)

print(
    "Test maps:",
    len(test_dataset),
    flush=True
)

Train maps: 12000
Validation maps: 1500
Test maps: 1500


# DataLoaders

In [13]:
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=g
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    generator=g
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    generator=g
)

# Model

In [14]:
model = CosmoCNN().to(
    device
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Training

In [15]:
best_val_loss = float("inf")

train_losses = []
val_losses = []


print(
    "\nStarting training...\n",
    flush=True
)


for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0


    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred = model(X)

        loss = criterion(
            pred,
            y
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item()
            * X.size(0)
        )


    train_loss /= len(
        train_loader.dataset
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0


    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            pred = model(X)

            loss = criterion(
                pred,
                y
            )

            val_loss += (
                loss.item()
                * X.size(0)
            )


    val_loss /= len(
        val_loader.dataset
    )


    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    train_losses.append(
        train_loss
    )

    val_losses.append(
        val_loss
    )


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    best_mark = ""

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        best_mark = " <-- best"


    # Save history every epoch
    # so results survive even if the job stops.
    np.savez(
        HISTORY_PATH,
        train_loss=np.array(
            train_losses
        ),
        val_loss=np.array(
            val_losses
        )
    )


    print(
        f"Epoch {epoch+1:3d} "
        f"Train {train_loss:.6f} "
        f"Val {val_loss:.6f}"
        f"{best_mark}",
        flush=True
    )


Starting training...

Epoch   1 Train 0.965114 Val 0.953666 <-- best
Epoch   2 Train 0.912166 Val 0.931913 <-- best
Epoch   3 Train 0.822153 Val 0.759671 <-- best
Epoch   4 Train 0.744340 Val 0.692586 <-- best
Epoch   5 Train 0.714649 Val 0.702438
Epoch   6 Train 0.702908 Val 0.659304 <-- best
Epoch   7 Train 0.681948 Val 0.639042 <-- best
Epoch   8 Train 0.665690 Val 0.644125
Epoch   9 Train 0.654262 Val 0.618370 <-- best
Epoch  10 Train 0.641156 Val 0.616199 <-- best
Epoch  11 Train 0.633288 Val 0.598848 <-- best
Epoch  12 Train 0.621956 Val 0.586739 <-- best


KeyboardInterrupt: 